In [2]:
pip install google-play-scraper requests


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
#!/usr/bin/env python3
"""Resolve App Store + Play Store IDs for UAE government apps."""

import time, requests
from google_play_scraper import search as gp_search

APPS = [
    # === FEDERAL ===
    "UAE PASS", "TAMM Abu Dhabi", "DubaiNow",
    "MOHRE UAE", "ICP UAE", "MOI UAE",
    "MOHAP UAE", "MOFA UAE", "MOJ UAE",
    "Ministry of Education UAE", "MOEC UAE",
    "Ministry of Energy and Infrastructure UAE",
    "Ministry of Community Development UAE",
    "FTA UAE", "EmaraTax", "Federal Tax Authority",
    "Emirates Post", "Etihad Water and Electricity",
    "UAE Weather", "NCM UAE", "Emirates Red Crescent",
    "Watani Al Emarat", "Hayyak", "Fazaa",
    "Emirates Digital Wallet", "Klip UAE",
    "UAE Armed Forces", "Emirates NBD Government",
    "Basher UAE", "Invest UAE", "Sanad UAE",

    # === DUBAI ===
    "Dubai Police", "RTA Dubai", "DEWA", "Dubai Municipality",
    "Dubai Health Authority", "Dubai Courts", "Dubai Customs",
    "Salik", "Nol Pay", "Dubai Land Department", "Madinati Dubai",
    "DXB Airport", "Dubai Airports",
    "Dubai REST", "Dubai Now Business",
    "Dubai Economy", "Invest in Dubai",
    "KHDA", "Dubai Sports Council",
    "Dubai Culture", "Dubai Public Prosecution",
    "DHA Doctor for Every Citizen", "Dubai Health",
    "Dubai Civil Defence", "Dubai Corporation for Ambulance Services",
    "Dubai Taxi", "Careem Dubai Metro", "S'hail",
    "Parkin Dubai", "Dubai Municipality 24x7",
    "Smart Salem", "Dubai Chamber", "DIFC",
    "Dubai Municipality Montaji", "Mahakim",

    # === ABU DHABI ===
    "Abu Dhabi Police", "ADDC", "Abu Dhabi Municipality",
    "Darb Abu Dhabi", "Abu Dhabi Judicial", "ADAFSA",
    "SEHA", "Abu Dhabi Public Health", "Malaffi",
    "ADEK Abu Dhabi", "Abu Dhabi Department of Education",
    "Abu Dhabi Sports Council", "Abu Dhabi Airports",
    "Mawaqif", "Abu Dhabi Distribution Company",
    "ADIO", "Abu Dhabi Chamber", "TAMM Business",
    "Abu Dhabi Civil Defence", "Abu Dhabi Judicial Department",
    "Al Ain Municipality", "Abu Dhabi Housing Authority",
    "Abu Dhabi Agriculture", "ADQ",

    # === SHARJAH ===
    "Sharjah Police", "Sharjah Municipality", "SEWA Sharjah",
    "Digital Sharjah", "Sharjah Courts",
    "Sharjah Roads and Transport", "Sharjah Airport",
    "Sharjah Chamber", "Sharjah Charity",
    "Sharjah Islamic Affairs", "Sharjah Social Services",

    # === AJMAN ===
    "Ajman Police", "Ajman Municipality", "AjmanOne",
    "Ajman DED", "Ajman Chamber", "Ajman Courts",
    "Ajman Transport",

    # === RAS AL KHAIMAH ===
    "RAK Police", "RAK Municipality", "mRak",
    "RAKTA", "RAK Courts", "RAK Chamber",
    "RAK Economic Zone", "RAK Hospital",

    # === FUJAIRAH ===
    "Fujairah Municipality", "Digital Fujairah",
    "Fujairah Police", "Fujairah Courts",

    # === UMM AL QUWAIN ===
    "UAQ Municipality", "SmartUAQ", "UAQ Police",
    "UAQ Free Trade Zone",

    # === UTILITIES / TRANSPORT / CROSS-EMIRATE ===
    "FEWA", "Etihad Rail", "Emirates Transport",
    "Etisalat Government", "Du Government",
    "Emirates Identity Authority", "Tasjeel",
    "Amer Dubai", "Tasheel UAE", "Shamil UAE",
]

def apple_lookup(term, country="ae"):
    r = requests.get("https://itunes.apple.com/search", params={
        "term": term, "country": country, "entity": "software", "limit": 3})
    out = []
    for x in r.json().get("results", []):
        out.append((x["trackId"], x["trackName"], x["sellerName"]))
    return out

def play_lookup(term, country="ae"):
    try:
        res = gp_search(term, lang="en", country=country, n_hits=3)
        return [(str(x.get("appId") or ""),
                 str(x.get("title") or ""),
                 str(x.get("developer") or "")) for x in res]
    except Exception as e:
        print(f"  [AND] error: {e}")
        return []

for app in APPS:
    print(f"\n{'='*60}\n{app}")
    for tid, name, dev in apple_lookup(app):
        print(f"  [iOS]  {tid:<12} {name[:40]:<42} {dev[:30]}")
    for pid, name, dev in play_lookup(app):
        print(f"  [AND]  {pid:<40} {name[:30]:<32} {dev[:25]}")
    time.sleep(0.5)


UAE PASS
  [iOS]  1377158818   UAE PASS                                   Dubai E-Government
  [iOS]  1374301965   UAEICP                                     Federal Authority for Identity
  [iOS]  768665731    MOI UAE                                    Ministry of Interior UAE
  [AND]                                           UAE PASS                         Digital Dubai Authority
  [AND]  com.echannels.moismartservices           UAEICP                           ICP Identity customs and 
  [AND]  com.uaemoi.smartservices                 MOI UAE                          Ministry of Interior U.A.

TAMM Abu Dhabi
  [iOS]  1435485576   TAMM - Abu Dhabi Government                Abu Dhabi Systems and Informat
  [iOS]  1377158818   UAE PASS                                   Dubai E-Government
  [iOS]  1374301965   UAEICP                                     Federal Authority for Identity
  [AND]                                           TAMM - Abu Dhabi Government      Department of Govern

In [10]:
import re, requests, time

def play_pkg(name):
    r = requests.get("https://play.google.com/store/search",
                     params={"q": name, "c": "apps", "hl": "en", "gl": "AE"},
                     headers={"User-Agent": "Mozilla/5.0"})
    return list(dict.fromkeys(re.findall(r'/store/apps/details\?id=([\w\.]+)', r.text)))[:5]

def ios_id(name):
    r = requests.get("https://itunes.apple.com/search",
                     params={"term": name, "country": "ae",
                             "entity": "software", "limit": 5}).json()
    return [(x["trackId"], x["trackName"], x["sellerName"]) for x in r["results"]]

MISSING_ANDROID = [
    "Esaad Card Dubai Police", "DEWA Dubai Electricity Water",
    "Al Munasiq Dubai Customs", "IDeclare Dubai Customs",
    "Dubai REST Dubai Land Department", "Smart Salik",
    "KHDA", "KHDA SmartApp", "Dubai Reads Civil Defence",
    "Dubai Culture", "Dubai Sports Council", "Dulook DXB",
    "Abu Dhabi Link Via", "AD Judicial Abu Dhabi",
    "Balligh Al Niyaba", "Smart Makani Abu Dhabi",
    "ADAFSA learning", "ADAFSA Water Allocation",
    "AD DOF Abu Dhabi Finance", "Bayaan Abu Dhabi Statistics",
    "CCAO Citizens Community Affairs", "SEWA Sharjah",
    "Sharjah Social Services", "Dawaei Sharjah", "SAIF ZONE Sharjah",
    "MPDA Ajman Municipality", "AJRC Ajman Courts",
    "RAK Police", "Fujairah Police", "Emirates Post",
    "Emirates Red Crescent", "OneET Emirates Transport",
    "Etihad Rail tickets", "Parkin Dubai", "Fazaa",
    "DIFC Plus", "DIFC Family Wealth Centre",
    "Dubai Chambers", "Sharjah Chamber of Commerce",
    "Volunteers.ae Emirates Foundation",
]

MISSING_IOS = [
    "RTA Smart Drive", "Build In Dubai", "Dubai Public Prosecution",
    "Dubai Trade", "RDC Dubai Land Department", "KHDA Wayfinding",
    "CDA Dubai Community Development", "Visit Dubai",
    "We Are All Police Abu Dhabi", "Healthcare Facility Audit DOH",
    "Abu Dhabi DOH TA", "TAQA Distribution", "OnwaniClick",
    "MyLand Abu Dhabi", "Tomouh", "Sharjah Public Prosecution",
    "DIAS Islamic Affairs Sharjah", "TAHSEEL Sharjah",
    "Bus On Demand Sharjah", "Ajman Police Club",
    "Ajman DED Inspection", "MAWARED Ajman", "Ajman Sewerage",
    "Fujairah Innovate", "UAQ DED Inspection",
    "MyDtc Dubai Taxi", "ADX Mobile", "ADX Investor",
    "iVestor Dubai Financial Market",
]

print("=== ANDROID ===")
for n in MISSING_ANDROID:
    try: print(n, play_pkg(n))
    except Exception as e: print(n, "ERR", e)
    time.sleep(1.5)

print("\n=== IOS ===")
for n in MISSING_IOS:
    try:
        print(f"\n{n}")
        for h in ios_id(n): print("   ", h)
    except Exception as e: print(n, "ERR", e)
    time.sleep(1)

=== ANDROID ===
Esaad Card Dubai Police ['dubaipolice.esaad.ae.esaad_dubaipolice', 'com.dubaipolice.app', 'com.dubaipolice.signlanguage', 'com.dubaipolice.cop28', 'ae.uaepass.mainapp']
DEWA Dubai Electricity Water ['com.dewa.application', 'com.fewa.eService', 'com.etihadwe.evcharging', 'com.sewa', 'duleaf.duapp.splash']
Al Munasiq Dubai Customs ['ae.gov.dubaicustoms.xclassifier.androidapp', 'ae.gov.dubaicustoms.mobile', 'doh.health.shield', 'ae.gov.dm.uma', 'com.uaemoi.smartservices']
IDeclare Dubai Customs ['ae.gov.dubaicustoms.mobile', 'ae.dubaicustoms.ideclare', 'com.echannels.moismartservices', 'com.license_dxb', 'com.dubaiculture']
Dubai REST Dubai Land Department ['ae.gov.dubailand.selfregistration', 'com.dhretechnology.dp', 'com.bayut.bayutapp', 'ae.gov.dubailand.rdc', 'ae.propertyfinder.propertyfinder']
Smart Salik ['com.salik.smartsalik', 'com.euroland.irapp.ae_salik', 'com.rta.rtadubai', 'com.itc.tollGate', 'com.dubaipolice.app']
KHDA ['ae.gov.khda', 'com.khda_rtls', 'com.kha

In [14]:
#!/usr/bin/env python3
"""
Scrape UAE government app reviews (App Store + Google Play)
into the target dataset schema.

pip install google-play-scraper requests
"""

import csv
import re
import time
import hashlib
from collections import Counter
import os


import requests

import requests
from google_play_scraper import reviews as gp_reviews, Sort

OUT_PATH = "reviews_dataset_big_more_20.csv"
CHECKPOINT_PATH = "scrape_checkpoint.txt"
CAP_PER_APP = 20000

FIELDS = [
    "review_id", "app_name", "platform", "review_text", "language",
    "star_rating", "review_date", "clean_text", "satisfaction_label",
    "aspect_tags",
]

# APPS = [
#     # ---- Federal ----
#     ("UAE PASS",              "ae.uaepass.mainapp",                  "1377158818"),
#     ("UAEICP",                "com.echannels.moismartservices",      "1374301965"),
#     ("MOI UAE",               "com.uaemoi.smartservices",            "768665731"),
#     ("MOHRE",                 "ae.gov.mol",                          "807379317"),
#     ("MOHAP",                 "com.gov.uae.mohap",                   "1462565560"),
#     ("MOE UAE",               "ae.gov.moe.mobileservices",           "1475400660"),
#     ("MOHESR UAE",            "ae.gov.mohesr.app",                   "6737803244"),
#     ("MoET UAE",              "ae.economy.MOEDashboards",            "1458324701"),
#     ("Ministry of Justice",   "com.timeline.mojmobile.ae",           "1163580188"),
#     ("UAE Public Prosecution","com.pp_smart_services_app",           None),
#     ("Emirates Health Svcs",  None,                                  "1163379015"),
#     ("Etihad WE (FEWA)",      "com.fewa.eService",                   "880606146"),

#     # ---- Dubai ----
#     ("DubaiNow",              "com.deg.mdubai",                      "619712783"),
#     ("Dubai Police",          "com.dubaipolice.app",                 "384374316"),
#     ("RTA Dubai",             "com.rta.rtadubai",                    "426109507"),
#     ("nol Pay",               "com.snowballtech.rta",                "1541976471"),
#     ("S'hail",                "com.rta.suhail",                      "1214681230"),
#     ("RTA Smart Drive",       "com.mireo.rtasmartdrive",             None),
#     ("DEWA",                  None,                                  "364928325"),
#     ("Dubai Municipality",    None,                                  "1504636184"),
#     ("Dubai Health (DAHC)",   "ae.gov.dha.flagship",                 "1437186269"),
#     ("DHA",                   None,                                  "6471334093"),
#     ("Dubai Courts",          None,                                  "946033161"),
#     ("Dubai Customs Munasiq", None,                                  "6504709282"),
#     ("IDeclare",              None,                                  "1542187927"),
#     ("Dubai Trade",           "ae.dubaitrade.dtmobile",              None),
#     ("Dubai REST (DLD)",      None,                                  "1437805105"),
#     ("Smart Salik",           None,                                  "912158362"),

#     # ---- Abu Dhabi ----
#     ("TAMM",                  "abudhabi.tamm.live",                  "1435485576"),
#     ("We Are All Police",     "wrplc.adpmb.com.weareallpolice",      None),
#     ("SEHA",                  "com.seha.app",                        "436297690"),
#     ("Sahatna",               "com.doh.sahatna",                     "6472413092"),
#     ("TAQA Distribution",     "com.ADDC.addcApp",                    "1045166599"),
#     ("DARB",                  "com.qmobility.darbx",                 "1509721720"),
#     ("Darbi",                 "com.dot.darbmobile",                  "840100351"),
#     ("AD Judicial",           "gov.adjd.Auctions",                   "6450068280"),
#     ("Iskan Abu Dhabi",       None,                                  "6443846523"),
#     ("Smart Makani",          None,                                  "1506588280"),
#     ("OnwaniClick",           "com.onwaniclick.dpm",                 None),
#     ("MyLand Abu Dhabi",      "com.myland.dpm",                      None),

#     # ---- Sharjah ----
#     ("Digital Sharjah",       "ae.sharjah.ds",                       "1569055813"),
#     ("RTA Sharjah",           "com.sharjahrta",                      "1187003188"),
#     ("Baladiyati Sharjah",    "com.sharjah.municipality",            "1584782736"),
#     ("Mawqef",                "shj.municipality.parking",            "6739573568"),
#     ("SEWA",                  None,                                  "721507762"),

#     # ---- Northern Emirates ----
#     ("Ajman Police",          "ae.gov.ajmanpolice.ajmanpolice",      "979481467"),
#     ("AjmanOne",              "com.Ajec",                            "1163528416"),
#     ("MPDA Ajman",            None,                                  "731268814"),
#     ("RAK Police",            None,                                  "1312657404"),
#     ("smartFUJAIRAH",         None,                                  "1373129135"),
#     ("IFujairah",             "gov.ae.ifujairah",                    "1178522130"),
#     ("DigitalUAQ",            "uae.gov.smartuaq",                    None),
# ]

APPS = [
    # ================= FEDERAL =================
    # ================= FEDERAL =================
    ("UAE PASS",                "ae.uaepass.mainapp",                    "1377158818"),
    ("UAEICP",                  "com.echannels.moismartservices",        "1374301965"),
    ("UAE Fast Track",          "com.fasttrack.uae.icp",                 "6476561074"),
    ("MOI UAE",                 "com.uaemoi.smartservices",              "768665731"),
    ("MOHRE",                   "ae.gov.mol",                            "807379317"),
    ("MOHRE G2G",               "ae.gov.mol.g2g",                        "1175881857"),
    ("MOHAP",                   "com.gov.uae.mohap",                     "1462565560"),
    ("Emirates Health Services", "com.mohgov.uae.MOHAPPP",               "1163379015"),
    ("MOE UAE",                 "ae.gov.moe.mobileservices",             "1475400660"),
    ("MOHESR UAE",              "ae.gov.mohesr.app",                     "6737803244"),
    ("MoET UAE",                "ae.economy.MOEDashboards",              "1458324701"),
    ("Ministry of Justice",     "com.timeline.mojmobile.ae",             "1163580188"),
    ("UAE Public Prosecution",  "com.pp_smart_services_app",             "1568248550"),
    ("UAE MOFA",                "com.mob.uae",                           "540714559"),
    ("MOFUAE (Finance)",        "ae.gov.mofuae",                         "981612710"),
    ("Ministry of Energy & Infra", "com.moenr.gov.ae",                   "892396014"),
    ("MOEI Wallet",             "com.stg.moei_wallet",                   "6670192941"),
    ("MOCE Employees",          "com.mocd.mocdapp",                      "1490376957"),
    ("EMARATAX",                "ae.gov.emaratax",                       "1660371526"),
    ("FTA Excise Connect",      "com.exciseconnect.eca.ae",              "1488610662"),
    ("Maskan - FTA",            "maskanrefund.tax.gov.ae",               "6478710219"),
    ("Tajneed (MOD)",           "ae.mod.tajneed",                        "6517355120"),
    ("AWQAF UAE",               "com.awqafuae.android",                  "1499083710"),
    ("UAE Weather (NCM)",       "com.uae.ncms",                          "497964984"),
    ("World Weather (NCM)",     "ae.ncm.android.worldweather",           None),
    ("UAE-Laws",                None,                                    "364916105"),
    ("InvestUAE Connect",       "ae.gov.invest.moiuae",                  "6759544509"),
    ("Etihad WE (FEWA)",        "com.fewa.eService",                     "880606146"),
    ("Etihad WE Consultant",    "com.etihadwe.consultant",               "1609231811"),
    ("Sanadak (Central Bank)",  "com.sanadak.sanadak",                   "6480274912"),

    # ================= DUBAI =================
    ("DubaiNow",                "com.deg.mdubai",                        "619712783"),
    ("Dubai Police",            "com.dubaipolice.app",                   "384374316"),
    ("Esaad Card",              "dubaipolice.esaad.ae.esaad_dubaipolice", "1475890066"),
    ("RTA Dubai",               "com.rta.rtadubai",                      "426109507"),
    ("nol Pay",                 "com.snowballtech.rta",                  "1541976471"),
    ("S'hail",                  "com.rta.suhail",                        "1214681230"),
    ("RTA Smart Drive",         "com.mireo.rtasmartdrive",               "926094022"),
    ("DEWA",                    "com.dewa.application",                  "364928325"),
    ("Dubai Municipality",      "ae.gov.dm.uma",                         "1504636184"),
    ("Destinations and more",   "ae.dubaipublicparks.prod",              "6450247865"),
    ("Build In Dubai",          "io.ionic.BPS",                          "1318091114"),
    ("Dubai Health (DAHC)",     "ae.gov.dha.flagship",                   "1437186269"),
    ("DHA",                     "ae.gov.dhamobile",                      "6471334093"),
    ("Dubai Courts",            "ae.gov.dcpetitions.mobile.iphone",      "946033161"),
    ("Dubai Public Prosecution", "ae.gov.dxbpp.smartprosecution",        None),
    ("Al Munasiq (Customs)",    "ae.gov.dubaicustoms.mobile",            "6504709282"),
    ("IDeclare",                "ae.dubaicustoms.ideclare",              "1542187927"),
    ("Dubai Trade",             "ae.dubaitrade.dtmobile",                "6503691030"),
    ("Dubai REST (DLD)",        "ae.gov.dubailand.selfregistration",     "1437805105"),
    ("RDC (DLD)",               "ae.gov.dubailand.rdc",                  "6469329089"),
    ("Smart Salik",             "com.salik.smartsalik",                  "912158362"),
    ("GDRFA DXB",               "ae.dnrd.gdrfad",                        "1625664521"),
    ("KHDA",                    "ae.gov.khda",                           "510571246"),
    ("KHDA SmartApp",           None,                                    "6753711684"),
    ("KHDA Wayfinding",         "com.khda_rtls",                         "6759160840"),
    ("Dubai Civil Defence",     "ae.gov.dcd.dlsp",                       "1255827772"),
    ("DCD Readiness",           "dcd.gov.ae.dcd_readiness",              "6443470364"),
    ("Dubai Reads",             None,                                    "1662905021"),
    ("Dubai Ambulance",         "com.dcas.app",                          "1165959800"),
    ("ESEFNI DCAS",             "dcas.esefni.app",                       "6464299386"),
    ("CDA Dubai",               "com.ionicframework.cdaapp133108",       "923357002"),
    ("CDA Sanad Relay",         "se.nwise.mmxtc.cda.sanadrelay",         "1504690367"),
    ("My Rights CDA",           None,                                    "1103161209"),
    ("Dubai Culture",           "com.dubaiculture",                      "926793557"),
    ("Dubai Library",           "com.dcaa.aas",                          "921365930"),
    ("Visit Dubai (DET)",       "com.dtcm.dubaitourism",                 "925400191"),
    ("Dubai Calendar (DET)",    None,                                    "501018460"),
    ("Dubai Sports Council",    "ae.gov.dsc",                            "6469217591"),
    ("IACAD Prayer Timings",    "ae.gov.iacad.salahtimingapp",           "1331933253"),

    # ================= ABU DHABI =================
    ("TAMM",                    "abudhabi.tamm.live",                    "1435485576"),
    ("We Are All Police",       "wrplc.adpmb.com.weareallpolice",        "1151447936"),
    ("SEHA",                    "com.linkdev.seha",                      "436297690"),
    ("Sahatna (DOH)",           "com.doh.sahatna",                       "6472413092"),
    ("Healthcare Facility Audit", "com.accela.acam_doh",                 "6471409247"),
    ("Abu Dhabi DOH TA",        "ae.gov.doh.stmobile",                   None),
    ("TAQA Distribution AD",    "com.addc.utilityapp",                   "1045166599"),
    ("TAQA Distribution Al Ain", "com.aadcsmartapp",                     "944770796"),
    ("DARB",                    "com.qmobility.darbx",                   "1509721720"),
    ("Darbi (ITC)",             "com.dot.darbmobile",                    "840100351"),
    ("Abu Dhabi Link",          None,                                    "1505305887"),
    ("AD Judicial",             "com.ADJD",                              "6450068280"),
    ("ADJD Auctions",           "gov.adjd.Auctions",                     "1614141284"),
    ("ADJD Complaints",         "gov.adjd.complaints",                   None),
    ("Balligh Al Niyaba",       None,                                    "1359049958"),
    ("Iskan Abu Dhabi",         "iskan.abudhabi.gov.ae",                 "6443846523"),
    ("Smart Makani",            "com.smartmakani.dmt",                   "1506588280"),
    ("OnwaniClick",             "com.onwaniclick.dpm",                   "1397189133"),
    ("MyLand Abu Dhabi",        "com.myland.dpm",                        "1459796069"),
    ("ADAFSA Self Inspection",  "com.selfinspection",                    "6443712930"),
    ("ADAFSA learning",         None,                                    "6467007675"),
    ("ADAFSA Water Allocation", None,                                    "6755325632"),
    ("ADCDA Community Responder", "com.adcda.communityresponder",        "6751804876"),
    ("Abu Dhabi 360 (ADSC)",    "com.adsc.abudhabi360",                  "6444888839"),
    ("AD DOF",                  "com.dof.app",                           "1463564835"),
    ("Bayaan (SCAD)",           "ae.gov.scad.bayaan.open",               "6499339305"),
    ("Tomouh (DGE)",            "com.dge.academy",                       "6736364399"),
    ("CCAO",                    None,                                    "960254035"),
    ("AUH Guest",               "ae.abudhabiairport.welcome",            "6468561918"),
    ("Experience Abu Dhabi",    "com.visitabudhabi.android",             "721678554"),

    # ================= SHARJAH =================
    ("Digital Sharjah",         "ae.sharjah.ds",                         "1569055813"),
    ("RTA Sharjah",             "com.sharjahrta",                        "1187003188"),
    ("Baladiyati Sharjah",      "com.sharjah.municipality",              "1584782736"),
    ("Mawqef Sharjah",          "shj.municipality.parking",              "6739573568"),
    ("SEWA",                    "com.sewa",                              "721507762"),
    ("SEWA MGR",                "com.sewa.sewamgr",                      "6461600959"),
    ("Sharjah Public Prosecution", "com.sharjah_pp_mobile_app",          None),
    ("Sharjah Social Services", None,                                    "1439799709"),
    ("Dawaei",                  None,                                    "1191274022"),
    ("DIAS (Islamic Affairs)",  "com.islamicaffairssharjah",             None),
    ("TAHSEEL Sharjah",         "gov.sfd.tahseelapp",                    "1493421998"),
    ("Bus On Demand Sharjah",   "com.liftango.busondemandsharjah",       "6654901688"),
    ("SAIF ZONE",               "com.saifzone.app",                      "1502229321"),
    ("SEDD Sharjah",            "ae.gov.sedd",                           None),

    # ================= AJMAN =================
    ("Ajman Police",            "ae.gov.ajmanpolice.ajmanpolice",        "979481467"),
    ("Ajman Police Club",       "com.vowalaa.ajman_app",                 "6463653002"),
    ("AjmanOne",                "com.Ajec",                              "1163528416"),
    ("MPDA Ajman",              "am.gov.ae.smartservices",               "731268814"),
    ("Ajman DED Inspection",    "si.ajman.ded.ae.siaded",                None),
    ("MAWARED Ajman",           "com.ajmanhrd",                          "6443741454"),
    ("Ajman Rulers Court",      "ae.ajman.rulerscourt",                  "6756175120"),
    ("Ajman VOD",               "ajman.rider",                           "1538316788"),
    ("Ajman Sewerage",          "com.moalajah.ajmansewerageutility",     "1158776017"),

    # ================= RAS AL KHAIMAH =================
    ("RAK Police",              None,                                    "1312657404"),
    ("mRAK",                    "ae.rak.ega.mrak",                       "767865884"),
    ("Sayr by RAKTA",           "ae.rakta.alhamrabus",                   "1525073858"),
    ("Albosala (RAKTA)",        "com.rakta.albosala",                    "1553878430"),
    ("RAKEZ",                   "com.rakez",                             "1581786595"),

    # ================= FUJAIRAH =================
    ("smartFUJAIRAH",           "gov.ae.fujmun.smartfujairah",           "1373129135"),
    ("IFujairah",               "gov.ae.ifujairah",                      "1178522130"),
    ("Digital Fujairah",        "com.egov.digitalfujairahapp",           "6744321237"),
    ("Fujairah Innovate",       "com.fuj_innovate",                      "1453851153"),
    ("Fujairah Police",         "com.FujairahPolice",                    "1555994726"),

    # ================= UMM AL QUWAIN =================
    ("DigitalUAQ",              "uae.gov.smartuaq",                      "1063110068"),
    ("UAQ DED Inspection",      "com.ded.inspection",                    None),

    # ================= SEMI-GOVERNMENT =================
    ("Emirates Post",           "ae.emiratespost.retailapp",             "6449459572"),
    ("EMX Express",             "ae.emiratespost.app",                   "1511692321"),
    ("Emirates Red Crescent",   "ae.rcuae.rcuae_app",                    "979176387"),
    ("Emirates Transport Booking", "dt.ptsfleetman.ftsservicepro.et",    "6504665528"),
    ("Emirates Transport UTS",  "com.emiratestransport.uts",             "1541525393"),
    ("ArKaNy (ET)",             "ae.et.HrMobApp",                        "1558147844"),
    ("Etihad Rail",             "com.etihadrail.app_prod_sds",           "6751881671"),
    ("Parkin",                  "parkin.ae.dev",                         "6657993734"),
    ("My DTC (Dubai Taxi)",     "com.dtc.mydtc",                         "6448499928"),
    ("Fazaa",                   "ae.fazaa",                              "1049790992"),
    ("DIFC+",                   "com.difc",                              "1282901202"),
    ("DIFC Family Wealth Centre", "com.mightybell.difc",                 "6758109642"),
    ("DFM",                     "com.DFM",                               "997641752"),
    ("ADX Mobile",              "ae.adx.mobile",                         "6504854914"),
    ("ADX Investor",            "com.directfn.universal_adx",            "1487431314"),
    ("iVestor (DFM)",           "ae.dfm.ivestorapp",                     "1628119841"),
    ("Dubai Chambers",          "com.app.dubaichamber",                  "780502711"),
    ("Ajman Chamber",           "ae.ajmanchamber.eservices",             "1591298239"),
    ("Sharjah Chamber",         None,                                    "922421821"),
    ("Sharjah Promotions",      "com.app.sharjahpromotions",             "6748413650"),
    ("Volunteers.ae",           "ae.emiratesfoundation.volunteer",       "1287025745"),
]
GP_LANGS = ("ar", "en")          # Arabic FIRST so it is never starved
GP_COUNTRIES = ("ae",)
AS_COUNTRIES = ("ae",)           # UAE storefront only

MIXED_THRESHOLD = 0.15           # minority-script share to call a review mixed

FIELDS = ["review_id", "app_name", "platform", "review_text", "language",
          "star_rating", "review_date", "clean_text", "satisfaction_label",
          "aspect_tags"]

ARABIC = re.compile(r"[\u0600-\u06FF]")
LATIN = re.compile(r"[A-Za-z]")
URL = re.compile(r"https?://\S+|www\.\S+")
DIACRITICS = re.compile(r"[\u064B-\u065F\u0670\u0640]")
EMOJI = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF]", flags=re.UNICODE)

# NOTE: keyword-based aspect tagging. Provenance must be documented — do NOT
# use this to validate BERTopic output in RQ3; run topic modelling independently.
ASPECTS = {
    "login": ["login", "log in", "sign in", "otp", "password", "uae pass",
              "authentication", "تسجيل الدخول", "كلمة المرور", "رمز"],
    "payment": ["payment", "pay", "fee", "card", "refund", "charge", "invoice",
                "دفع", "رسوم", "بطاقة", "استرداد", "فاتورة"],
    "usability": ["easy", "confusing", "interface", "ui", "ux", "navigate",
                  "design", "simple", "سهل", "واجهة", "معقد", "تصميم"],
    "performance": ["slow", "crash", "freeze", "lag", "loading", "bug", "error",
                    "بطيء", "يتوقف", "تعليق", "خطأ", "بطء"],
    "update": ["update", "version", "upgrade", "new version",
               "تحديث", "الاصدار", "النسخة"],
    "support": ["support", "customer service", "help", "contact", "response",
                "دعم", "خدمة العملاء", "مساعدة", "تواصل"],
    "verification": ["verify", "verification", "emirates id", "document", "upload",
                     "تحقق", "الهوية", "مستند", "رفع"],
    "notifications": ["notification", "alert", "reminder", "sms",
                      "اشعار", "إشعار", "تنبيه", "رسالة"],
    "language": ["arabic", "english", "translation", "language",
                 "عربي", "العربية", "انجليزي", "ترجمة", "لغة"],
    "registration": ["register", "registration", "sign up", "account creation",
                     "تسجيل", "حساب جديد", "انشاء حساب"],
}

# ---------------------------------------------------------------- helpers


def detect_language(text):
    """Script-ratio language detection. 'mixed' requires a real minority share."""
    ar = len(ARABIC.findall(text or ""))
    la = len(LATIN.findall(text or ""))
    if ar + la == 0:
        return "unknown"
    if min(ar, la) / (ar + la) >= MIXED_THRESHOLD:
        return "mixed"
    return "Arabic" if ar > la else "English"


def normalise(text):
    t = URL.sub(" ", text or "")
    t = EMOJI.sub(" ", t)
    t = DIACRITICS.sub("", t)
    t = (t.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
          .replace("ى", "ي").replace("ة", "ه").replace("ؤ", "و")
          .replace("ئ", "ي"))
    t = re.sub(r"[^\w\s\u0600-\u06FF]", " ", t)
    return re.sub(r"\s+", " ", t).strip().lower()


def label_from_stars(stars):
    if not stars:
        return ""
    if stars >= 4:
        return "Satisfied"
    if stars == 3:
        return "Neutral"
    return "Dissatisfied"


def tag_aspects(clean, raw):
    blob = f"{clean} {(raw or '').lower()}"
    return "|".join(a for a, kws in ASPECTS.items() if any(k in blob for k in kws))


def make_row(app_name, platform, text, stars, date, uid):
    text = (text or "").strip()
    clean = normalise(text)
    return {
        "review_id": hashlib.md5(
            f"{platform}:{app_name}:{uid}".encode()).hexdigest()[:16],
        "app_name": app_name,
        "platform": platform,
        "review_text": text,
        "language": detect_language(text),
        "star_rating": stars if stars else "",
        "review_date": date,
        "clean_text": clean,
        "satisfaction_label": label_from_stars(stars),
        "aspect_tags": tag_aspects(clean, text),
    }

# ---------------------------------------------------------------- scrapers


def scrape_google_play(app_name, pkg, budget):
    """Each language gets its own budget so the Arabic pass is never skipped."""
    rows, seen_text = [], set()
    per_lang = max(budget // len(GP_LANGS), 1)

    for lang in GP_LANGS:
        got = 0
        for country in GP_COUNTRIES:
            token = None
            while got < per_lang:
                try:
                    batch, token = gp_reviews(
                        pkg, lang=lang, country=country, sort=Sort.NEWEST,
                        count=200, continuation_token=token)
                except Exception as e:
                    print(f"  [GP] {app_name} {lang}/{country} error: {e}")
                    break
                if not batch:
                    break

                for r in batch:
                    if got >= per_lang:
                        break
                    try:
                        row = make_row(
                            app_name, "Google Play", r.get("content"),
                            r.get("score"),
                            r["at"].date().isoformat() if r.get("at") else "",
                            r.get("reviewId"))
                    except Exception:
                        continue
                    if not row["review_text"] or row["clean_text"] in seen_text:
                        continue
                    seen_text.add(row["clean_text"])
                    rows.append(row)
                    got += 1

                print(f"  [GP] {app_name} {lang}/{country} -> {got}/{per_lang}")
                if token is None:
                    break
                time.sleep(0.3)
    return rows


def scrape_app_store(app_name, app_id, budget):
    """Dedups on text across storefronts — Apple reissues IDs per country."""
    rows, seen_text = [], set()

    for c in AS_COUNTRIES:
        for page in range(1, 11):
            url = (f"https://itunes.apple.com/{c}/rss/customerreviews/"
                   f"page={page}/id={app_id}/sortby=mostrecent/json")
            try:
                feed = requests.get(url, timeout=20).json().get("feed", {})
                entries = feed.get("entry", [])
            except Exception as e:
                print(f"  [AS] {app_name} {c} p{page} error: {e}")
                break

            if isinstance(entries, dict):      # Apple returns a bare dict for 1 entry
                entries = [entries]
            if not isinstance(entries, list) or not entries:
                break

            if page == 1 and isinstance(entries[0], dict) and "im:name" in entries[0]:
                entries = entries[1:]          # first entry is app metadata

            for e in entries:
                if len(rows) >= budget:
                    break
                if not isinstance(e, dict):
                    continue
                try:
                    text = f"{e['title']['label']}. {e['content']['label']}"
                    row = make_row(app_name, "App Store", text,
                                   int(e["im:rating"]["label"]),
                                   e["updated"]["label"][:10], e["id"]["label"])
                except (KeyError, TypeError, ValueError):
                    continue
                if not row["review_text"] or row["clean_text"] in seen_text:
                    continue
                seen_text.add(row["clean_text"])
                rows.append(row)

            print(f"  [AS] {app_name} {c} p{page} -> {len(rows)}/{budget}")
            if len(rows) >= budget:
                return rows
            time.sleep(0.4)
    return rows

# ---------------------------------------------------------------- checkpoint


def load_checkpoint():
    if not os.path.exists(CHECKPOINT_PATH):
        return set()
    with open(CHECKPOINT_PATH, encoding="utf-8") as f:
        return {line.strip() for line in f if line.strip()}


def mark_done(app_name):
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as f:
        f.write(app_name + "\n")


def load_existing_keys(path):
    """Rebuild dedup sets from a partial CSV so a resumed run stays consistent."""
    ids, texts = set(), set()
    if not os.path.exists(path):
        return ids, texts
    with open(path, encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            ids.add(r["review_id"])
            texts.add(r["clean_text"])
    return ids, texts

# ---------------------------------------------------------------- summary


def summarise(path=OUT_PATH):
    with open(path, encoding="utf-8-sig") as f:
        data = list(csv.DictReader(f))

    print("\n--- per app ---")
    for app, n in Counter(r["app_name"] for r in data).most_common():
        print(f"  {app:<26} {n}")

    print("\n--- distributions ---")
    print("platform:", dict(Counter(r["platform"] for r in data)))
    print("language:", dict(Counter(r["language"] for r in data)))
    print("label:   ", dict(Counter(r["satisfaction_label"] for r in data)))

    aspect_counts = Counter()
    for r in data:
        for a in filter(None, r["aspect_tags"].split("|")):
            aspect_counts[a] += 1
    print("aspects: ", dict(aspect_counts.most_common()))

    untagged = sum(1 for r in data if not r["aspect_tags"])
    print(f"untagged: {untagged} ({untagged / max(len(data), 1):.0%})")

    texts = [r["clean_text"] for r in data]
    print(f"unique texts: {len(set(texts))} / {len(texts)}")
    return data

# ---------------------------------------------------------------- main


def main():
    done = load_checkpoint()
    seen_ids, seen_texts = load_existing_keys(OUT_PATH)
    resuming = bool(done)
    total = len(seen_ids)

    if resuming:
        print(f"Resuming: {len(done)} apps done, {total} rows on disk.")

    mode = "a" if resuming else "w"
    with open(OUT_PATH, mode, newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if not resuming:
            writer.writeheader()

        for name, pkg, asid in APPS:
            if name in done:
                print(f"--- skipping {name} (already done)")
                continue

            print(f"\n=== {name} ===")
            app_rows = []

            if pkg:
                try:
                    app_rows += scrape_google_play(name, pkg, CAP_PER_APP)
                except Exception as e:
                    print(f"  !! GP failed for {name}: {e}")

            remaining = CAP_PER_APP - len(app_rows)
            if asid and remaining > 0:
                try:
                    app_rows += scrape_app_store(name, asid, remaining)
                except Exception as e:
                    print(f"  !! AS failed for {name}: {e}")

            fresh = 0
            for r in app_rows:
                if (r["review_id"] in seen_ids
                        or r["clean_text"] in seen_texts
                        or not r["review_text"]):
                    continue
                seen_ids.add(r["review_id"])
                seen_texts.add(r["clean_text"])
                writer.writerow(r)
                fresh += 1

            f.flush()
            mark_done(name)
            total += fresh
            print(f"  == {name}: {fresh} written (running total {total})")

    print(f"\nSaved {total} reviews -> {OUT_PATH}")
    summarise()


if __name__ == "__main__":
    main()


=== UAE PASS ===
  [GP] UAE PASS ar/ae -> 110/10000
  [GP] UAE PASS ar/ae -> 223/10000
  [GP] UAE PASS ar/ae -> 358/10000
  [GP] UAE PASS ar/ae -> 492/10000
  [GP] UAE PASS ar/ae -> 615/10000
  [GP] UAE PASS ar/ae -> 735/10000
  [GP] UAE PASS ar/ae -> 856/10000
  [GP] UAE PASS ar/ae -> 956/10000
  [GP] UAE PASS ar/ae -> 1051/10000
  [GP] UAE PASS ar/ae -> 1173/10000
  [GP] UAE PASS ar/ae -> 1287/10000
  [GP] UAE PASS ar/ae -> 1411/10000
  [GP] UAE PASS ar/ae -> 1490/10000
  [GP] UAE PASS en/ae -> 116/10000
  [GP] UAE PASS en/ae -> 235/10000
  [GP] UAE PASS en/ae -> 337/10000
  [GP] UAE PASS en/ae -> 460/10000
  [GP] UAE PASS en/ae -> 562/10000
  [GP] UAE PASS en/ae -> 677/10000
  [GP] UAE PASS en/ae -> 805/10000
  [GP] UAE PASS en/ae -> 937/10000
  [GP] UAE PASS en/ae -> 1050/10000
  [GP] UAE PASS en/ae -> 1174/10000
  [GP] UAE PASS en/ae -> 1309/10000
  [GP] UAE PASS en/ae -> 1440/10000
  [GP] UAE PASS en/ae -> 1558/10000
  [GP] UAE PASS en/ae -> 1679/10000
  [GP] UAE PASS en/ae -> 1

In [16]:
import pandas as pd, numpy as np, re

df = pd.read_csv("reviews_dataset_big_more_20.csv")   # adjust filename
df["n_words"] = df["review_text"].fillna("").str.split().str.len()
bands = pd.cut(df.n_words, [-1,2,5,10,25,10**6], labels=["0-2","3-5","6-10","11-25","26+"])

print("="*60, "\n1. TOTALS\n", "="*60)
print(f"rows: {len(df)}   cols: {list(df.columns)}")

print("\n", "="*60, "\n2. LANGUAGE x LENGTH (row %)\n", "="*60)
print((pd.crosstab(df.language, bands, normalize="index")*100).round(1))

print("\n", "="*60, "\n3. LANGUAGE x LABEL (counts + row %)\n", "="*60)
lx = pd.crosstab(df.language, df.satisfaction_label)
print(lx, "\n")
print((lx.div(lx.sum(1), axis=0)*100).round(1))

print("\n", "="*60, "\n4. ARABIC SURVIVAL AT LENGTH FILTERS\n", "="*60)
for k in [3, 5, 6, 8, 10]:
    s = df[df.n_words >= k]
    v = s.language.value_counts()
    ar, en = v.get("Arabic",0), v.get("English",0)
    print(f"  >={k:2d}w: n={len(s):6d}  AR={ar:6d} ({ar/max(len(s),1):.1%})  "
          f"EN={en:6d}  AR kept={ar/max((df.language=='Arabic').sum(),1):.1%}")

print("\n", "="*60, "\n5. NEUTRAL BY LANGUAGE (the H1 risk)\n", "="*60)
for lang in df.language.dropna().unique():
    s = df[df.language==lang]
    n = (s.satisfaction_label=="Neutral").sum()
    print(f"  {lang:10s} Neutral={n:5d} / {len(s):6d} = {n/max(len(s),1):.1%}")

print("\n", "="*60, "\n6. PER-APP BALANCE (top 25 + tail)\n", "="*60)
ac = df.app_name.value_counts()
print(f"  apps: {len(ac)}   top-5 share: {ac.head(5).sum()/len(df):.1%}   "
      f"apps with <100 reviews: {(ac<100).sum()}")
print(ac.head(25))
print("\n  Arabic count by app (top 15):")
print(df[df.language=="Arabic"].app_name.value_counts().head(15))

print("\n", "="*60, "\n7. TEMPORAL COVERAGE\n", "="*60)
d = pd.to_datetime(df.review_date, errors="coerce")
print(f"  parsed: {d.notna().mean():.1%}   range: {d.min()} -> {d.max()}")
print(d.dt.year.value_counts().sort_index())

print("\n", "="*60, "\n8. DUPLICATES\n", "="*60)
t = df.review_text.fillna("").str.strip().str.lower()
print(f"  exact dup texts: {t.duplicated().sum()} ({t.duplicated().mean():.1%})")
print(f"  unique texts: {t.nunique()}")
print("  most repeated:"); print(t[t.str.len()>0].value_counts().head(10))

print("\n", "="*60, "\n9. CODE-SWITCHING: detector vs script rule\n", "="*60)
has_ar = df.review_text.fillna("").str.contains(r"[\u0600-\u06FF]")
has_la = df.review_text.fillna("").str.contains(r"[A-Za-z]{2,}")
script_mixed = has_ar & has_la
print(f"  detector says 'mixed': {(df.language=='mixed').sum()}")
print(f"  script rule says mixed: {script_mixed.sum()}")
print(f"  script-mixed AND >=6 words: {(script_mixed & (df.n_words>=6)).sum()}")
print("\n  what the detector labelled script-mixed rows as:")
print(df.loc[script_mixed, "language"].value_counts())

print("\n", "="*60, "\n10. ASPECT COVERAGE BY LANGUAGE & LENGTH\n", "="*60)
tagged = df.aspect_tags.notna() & (df.aspect_tags.astype(str).str.strip()!="")
print("  by language:"); print(tagged.groupby(df.language).mean().round(3))
print("  by length:");   print(tagged.groupby(bands).mean().round(3))

print("\n", "="*60, "\n11. STAR -> LABEL MAPPING CHECK\n", "="*60)
if "star_rating" in df.columns:
    print(pd.crosstab(df.star_rating, df.satisfaction_label))
    print(f"\n  missing stars: {df.star_rating.isna().sum()}")

print("\n", "="*60, "\n12. BALANCED-SAMPLE FEASIBILITY\n", "="*60)
cells = df[df.language.isin(["Arabic","English"])].groupby(
    ["language","satisfaction_label", bands]).size().unstack(fill_value=0)
print(cells)
ar_min = df[(df.language=="Arabic")].groupby("satisfaction_label").size()
print(f"\n  Arabic per-class ceiling: {dict(ar_min)}")
print(f"  => max balanced 2-lang corpus at parity: {2*ar_min.sum()}")

1. TOTALS
rows: 74412   cols: ['review_id', 'app_name', 'platform', 'review_text', 'language', 'star_rating', 'review_date', 'clean_text', 'satisfaction_label', 'aspect_tags', 'n_words']

2. LANGUAGE x LENGTH (row %)
n_words    0-2   3-5  6-10  11-25   26+
language                               
Arabic    10.3  26.2  27.3   26.5   9.7
English    6.6  20.0  23.9   30.3  19.2
mixed      4.5  21.6  24.6   25.6  23.6
unknown   62.7  17.9   4.5   13.4   1.5

3. LANGUAGE x LABEL (counts + row %)
satisfaction_label  Dissatisfied  Neutral  Satisfied
language                                            
Arabic                      6587      855      12289
English                    25665     3249      25302
mixed                        173       22        203
unknown                       18        5         44 

satisfaction_label  Dissatisfied  Neutral  Satisfied
language                                            
Arabic                      33.4      4.3       62.3
English                   

/var/folders/dq/1sqw44_x10dfy5fspvw8hrm00000gn/T/ipykernel_35618/1168901598.py:64: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print("  by length:");   print(tagged.groupby(bands).mean().round(3))
/var/folders/dq/1sqw44_x10dfy5fspvw8hrm00000gn/T/ipykernel_35618/1168901598.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cells = df[df.language.isin(["Arabic","English"])].groupby(


In [18]:
import pandas as pd
df = pd.read_csv("reviews_dataset_big_more_20.csv")
df["n_words"] = df["review_text"].fillna("").str.split().str.len()

print(df["n_words"].describe())
print(df["n_words"].quantile([.25,.5,.75,.9,.95]))
for lo, hi in [(0,2),(3,5),(6,10),(11,25),(26,10**6)]:
    sub = df[df.n_words.between(lo,hi)]
    print(f"{lo}-{hi} words: {len(sub)} ({len(sub)/len(df):.0%})  "
          f"aspect-tagged: {(sub.aspect_tags.notna() & (sub.aspect_tags!='')).mean():.0%}")

# how much survives a substance filter, by language
sub = df[df.n_words >= 6]
print(sub.groupby("language").size())
print(sub.groupby(["language","satisfaction_label"]).size().unstack(fill_value=0))

count    74412.000000
mean        15.210813
std         16.972498
min          0.000000
25%          5.000000
50%          9.000000
75%         19.000000
max        657.000000
Name: n_words, dtype: float64
0.25     5.0
0.50     9.0
0.75    19.0
0.90    34.0
0.95    48.0
Name: n_words, dtype: float64
0-2 words: 5665 (8%)  aspect-tagged: 9%
3-5 words: 16107 (22%)  aspect-tagged: 25%
6-10 words: 18449 (25%)  aspect-tagged: 40%
11-25 words: 21787 (29%)  aspect-tagged: 58%
26-1000000 words: 12404 (17%)  aspect-tagged: 78%
language
Arabic     12526
English    39807
mixed        294
unknown       13
dtype: int64
satisfaction_label  Dissatisfied  Neutral  Satisfied
language                                            
Arabic                      5045      657       6824
English                    22145     2746      14916
mixed                        154       18        122
unknown                        6        2          5


In [5]:

import requests
r = requests.get("https://bayanat.ae/api/DatasetResources/GetDatasetResource",
                 params={"resourceID": "{ResourceGUID}"})

In [6]:
print(df.groupby("language").size())
print(df.groupby(["language", "satisfaction_label"]).size().unstack(fill_value=0))
print(pd.crosstab(df.language, pd.cut(df.n_words, [-1,2,5,10,25,10**6],
      labels=["0-2","3-5","6-10","11-25","26+"]), normalize="index"))

language
Arabic      6269
English    45469
mixed        189
unknown      571
dtype: int64
satisfaction_label  Dissatisfied  Neutral  Satisfied
language                                            
Arabic                      1505      173       4591
English                    13477     1907      30085
mixed                         83       15         91
unknown                       43       16        512
n_words        0-2       3-5      6-10     11-25       26+
language                                                  
Arabic    0.440900  0.204658  0.163822  0.143723  0.046897
English   0.379511  0.200422  0.148519  0.171699  0.099848
mixed     0.047619  0.089947  0.153439  0.365079  0.343915
unknown   0.961471  0.021016  0.007005  0.008757  0.001751
